In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
from catboost import CatBoostRegressor, Pool
import warnings
warnings.filterwarnings('ignore')

# ========================
# 1. ĐỌC DỮ LIỆU
# ========================
df = pd.read_csv('vietnam_housing_dataset - Copy.csv')  # thay bằng đường dẫn file thực tế của bạn

print(f"Shape ban đầu: {df.shape}")

# Chuẩn hóa tên cột
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

# Chuyển đổi kiểu dữ liệu
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['area'] = pd.to_numeric(df['area'], errors='coerce')

df = df.dropna(subset=['price', 'area'])
df = df[(df['price'] > 0) & (df['area'] > 0)]

# ========================
# 2. CLIP OUTLIER (rất quan trọng để giảm lệch lớn)
# ========================
lower_p, upper_p = df['price'].quantile([0.01, 0.99])
df['price'] = df['price'].clip(lower_p, upper_p)

lower_a, upper_a = df['area'].quantile([0.005, 0.995])
df['area'] = df['area'].clip(lower_a, upper_a)

print(f"Sau clip outlier: {df.shape[0]} dòng")

# ========================
# 3. EXTRACT CITY & DISTRICT (cải tiến chi tiết hơn)
# ========================
def extract_district_city_v2(address):
    if pd.isna(address) or not isinstance(address, str):
        return 'Other', 'Other'
    
    address_lower = address.lower().strip()
    
    # City mapping (giữ nguyên + mở rộng một chút)
    city_map = {
        'hồ chí minh': 'HCM', 'tp.hồ chí minh': 'HCM', 'tphcm': 'HCM', 'hcm': 'HCM',
        'hà nội': 'HN', 'thành phố hà nội': 'HN', 'hn': 'HN',
        'hưng yên': 'HY', 'bình dương': 'BD', 'đà nẵng': 'ĐN', 'đn': 'ĐN',
        'long an': 'LA', 'đồng nai': 'Đồng Nai', 'bà rịa vũng tàu': 'BRVT',
        'quảng ninh': 'QN', 'phú thọ': 'PT', 'hải phòng': 'HP', 'khánh hòa': 'KH',
        'kiên giang': 'KG', 'bình thuận': 'BT', 'hải dương': 'HD', 'thanh hóa': 'TH',
    }
    city = 'Other'
    for k, v in city_map.items():
        if k in address_lower:
            city = v
            break
    
    # Tìm district/quận/huyện bằng từ khóa
    district_keywords = ['quận ', 'huyện ', 'thị trấn ', 'phường ', 'xã ', 'thị xã ', 'tp.', 'thành phố ']
    district = 'Other'
    
    # Tách theo dấu phẩy
    parts = [p.strip() for p in address.split(',')]
    
    for part in parts:
        part_lower = part.lower()
        # Nếu phần này chứa từ khóa quận/huyện/phường/xã → coi là district
        if any(kw in part_lower for kw in district_keywords):
            # Lấy tên sạch hơn
            for kw in district_keywords:
                part = part.replace(kw, '').strip()
            district = part.title()
            break
        # Nếu không có từ khóa nhưng là tên quận/huyện quen thuộc
        elif any(d in part_lower for d in ['gò vấp', 'bình thạnh', 'quận 7', 'thủ đức', 'long biên', 'cầu giấy', 'đống đa', 'thanh xuân']):
            district = part.title()
            break
    
    # Fix tên quận/huyện phổ biến
    district_fix = {
        'gò vấp': 'Gò Vấp', 'bình thạnh': 'Bình Thạnh', 'quận 7': 'Quận 7',
        'thủ đức': 'Thủ Đức', 'long biên': 'Long Biên', 'cầu giấy': 'Cầu Giấy',
        'đống đa': 'Đống Đa', 'thanh xuân': 'Thanh Xuân', 'hà đông': 'Hà Đông',
        'nam từ liêm': 'Nam Từ Liêm', 'tây hồ': 'Tây Hồ', 'hoàn kiếm': 'Hoàn Kiếm',
        'bắc từ liêm': 'Bắc Từ Liêm', 'hoàng mai': 'Hoàng Mai', 'hai bà trưng': 'Hai Bà Trưng',
    }
    district_lower = district.lower()
    for k, v in district_fix.items():
        if k in district_lower:
            district = v
            break
    
    if district == 'Other' and 'dự án' in address_lower:
        # Với dự án lớn, thường có tên quận/huyện trong tên dự án
        if 'ocean park' in address_lower:
            district = 'Văn Giang'
        elif 'grand park' in address_lower:
            district = 'Quận 9'
        elif 'vinhomes' in address_lower and 'ocean' in address_lower:
            district = 'Văn Giang'
        # thêm rule nếu bạn thấy nhiều dự án cụ thể
    
    return city, district


# Thay thế hàm cũ bằng hàm mới
df['city'], df['district'] = zip(*df['address'].apply(extract_district_city_v2))

print("\nPhân bố City sau extract:")
print(df['city'].value_counts().head(15))
print("\nPhân bố District sau extract (top 20):")
print(df['district'].value_counts().head(20))

# ========================
# 4. FEATURE ENGINEERING
# ========================
df['log_price'] = np.log1p(df['price'])

# Impute numeric
numeric_cols = ['area', 'frontage', 'access_road', 'floors', 'bedrooms', 'bathrooms']
for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(df[col].median())

df['total_rooms']     = df['bedrooms'].fillna(0) + df['bathrooms'].fillna(0) + 1
df['bed_bath_ratio']  = df['bedrooms'] / (df['bathrooms'] + 1e-6)
df['area_per_floor']  = df['area'] / (df['floors'] + 1e-6)
df['rooms_per_floor'] = df['total_rooms'] / (df['floors'] + 1e-6)

good_dirs = ['Nam', 'Đông - Nam', 'Đông', 'Đông Nam']
df['is_good_direction'] = df['house_direction'].isin(good_dirs).astype(int)

# District avg price/m² (feature cực mạnh)
df['price_per_m2'] = df['price'] / df['area']
district_avg = df.groupby('district')['price_per_m2'].mean().to_dict()
df['district_avg_price_per_m2'] = df['district'].map(district_avg).fillna(df['price_per_m2'].median())

# Feature interaction
df['area_x_district_avg'] = df['area'] * df['district_avg_price_per_m2']
df['frontage_x_access']   = df['frontage'] * df['access_road']

# Categorical
cat_cols = ['house_direction', 'balcony_direction', 'legal_status', 
            'furniture_state', 'city', 'district']

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna('Missing').astype('category')

# ========================
# 5. CHUẨN BỊ FEATURES (KHÔNG leak price_per_m2)
# ========================
features = [
    'area', 'frontage', 'access_road', 'floors', 'bedrooms', 'bathrooms',
    'total_rooms', 'bed_bath_ratio', 'area_per_floor', 'rooms_per_floor',
    'is_good_direction', 'district_avg_price_per_m2',
    'area_x_district_avg', 'frontage_x_access',
    'house_direction', 'balcony_direction', 'legal_status', 'furniture_state',
    'city', 'district'
]

X = df[features].copy()
y = df['log_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

print(f"\nTrain: {len(X_train):,} | Test: {len(X_test):,}")

# ========================
# 6. TRAIN CATBOOST
# ========================
cat_features = [f for f in features if f in cat_cols]

train_pool = Pool(X_train, y_train, cat_features=cat_features)
test_pool  = Pool(X_test, y_test, cat_features=cat_features)

model = CatBoostRegressor(
    iterations=7000,
    learning_rate=0.012,
    depth=8,
    l2_leaf_reg=6,
    random_seed=42,
    loss_function='RMSE',
    eval_metric='MAE',
    early_stopping_rounds=180,
    verbose=200,
    use_best_model=True
)

model.fit(
    train_pool,
    eval_set=test_pool,
)


Shape ban đầu: (30229, 12)
Sau clip outlier: 30229 dòng

Phân bố City sau extract:
city
HCM         11785
HN          10459
BD           1675
ĐN           1450
Other        1047
Đồng Nai      844
HP            776
KH            725
HY            404
LA            341
BRVT          240
BT            127
QN            111
TH            110
KG             83
Name: count, dtype: int64

Phân bố District sau extract (top 20):
district
Phường 5                  449
Phường 11                 439
Phường 12                 439
Phường 14                 385
Phường 13                 340
Phường 15                 336
Phường 3                  330
Phường 10                 321
Xã Long Hưng              287
Phường 4                  275
Phường 7                  247
Phường 8                  237
Phường 6                  231
Phường 9                  231
Phường Tân Quý            222
Phường 1                  218
Phường Tân Đông Hiệp      215
Phường Phú Hữu            210
Phường Thạch Bàn          2

CatBoostRegressor(depth=8, early_stopping_rounds=180, eval_metric='MAE', iterations=7000, l2_leaf_reg=6, learning_rate=0.012, loss_function='RMSE', random_seed=42, use_best_model=True, verbose=200)

In [2]:
# ========================
# 7. ĐÁNH GIÁ
# ========================
y_pred_log = model.predict(X_test)

print("\n=== Log Price Metrics ===")
print(f"MAE:  {mean_absolute_error(y_test, y_pred_log):.5f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred_log)):.5f}")
print(f"R²:   {r2_score(y_test, y_pred_log):.5f}")

y_test_real  = np.expm1(y_test)
y_pred_real  = np.expm1(y_pred_log)

mae_real  = mean_absolute_error(y_test_real, y_pred_real)
rmse_real = np.sqrt(mean_squared_error(y_test_real, y_pred_real))
r2_real   = r2_score(y_test_real, y_pred_real)

print("\n=== Giá thật (tỷ VND) ===")
print(f"MAE:  {mae_real:,.2f} tỷ")
print(f"RMSE: {rmse_real:,.2f} tỷ")
print(f"R²:   {r2_real:.4f}")

# ========================
# 8. FEATURE IMPORTANCE
# ========================
fi = pd.DataFrame({
    'feature': features,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False).head(15)

print("\n=== Top 15 features quan trọng nhất ===")
print(fi.to_string(index=False))


=== Log Price Metrics ===
MAE:  0.13309
RMSE: 0.18191
R²:   0.74172

=== Giá thật (tỷ VND) ===
MAE:  0.89 tỷ
RMSE: 1.22 tỷ
R²:   0.6926

=== Top 15 features quan trọng nhất ===
                  feature  importance
      area_x_district_avg   36.178949
                     city    9.686078
                 district    7.667577
              access_road    5.506510
district_avg_price_per_m2    5.149340
                   floors    4.371850
        frontage_x_access    4.045666
              total_rooms    3.999831
                bathrooms    3.793430
                     area    3.401506
           area_per_floor    2.950841
          rooms_per_floor    2.467614
             legal_status    2.411813
          furniture_state    1.910742
                 frontage    1.880473
